# Grid detection training and evaluation
Run All trains and evaluates a one-epoch smoke run. Generate the selected dataset first; the first model load may download pretrained weights. For an existing run, select its `RUN_NAME`, run Setup and Configuration, skip Verify dataset and Train model, then run Load saved run onward. Validation supports model selection; reserve test for held-out assessment.

## 1. Setup

In [ ]:
import csv
import importlib.util
import json
import hashlib
import math
import platform
import random
from itertools import combinations
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from matplotlib.patches import Patch, Rectangle
from PIL import Image
import torch
import ultralytics
import yaml
from ultralytics import YOLO
from ultralytics.cfg import get_cfg
from ultralytics.utils.torch_utils import TORCH_2_0
from grid_detection.dataset.yolo import YOLO_CLASSES
from grid_detection.research import DATA_DIR, PROJECT_ROOT, RUNS_DIR
from synthetic_scenes.provenance import run_provenance

print(f"Python {platform.python_version()} | PyTorch {torch.__version__} | Ultralytics {ultralytics.__version__}")
print(f"Datasets: {DATA_DIR}")
print(f"Runs: {RUNS_DIR}")

# Identity uses content hashes for the checkpoint and dataset descriptors, plus
# the selected split inventory (size and modification time) for large files.
def file_hash(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def inventory(paths, root):
    return [[str(p.relative_to(root)), p.stat().st_size, p.stat().st_mtime_ns]
            for p in sorted(paths) if p.is_file()]

def read_completed(destination, expected, required):
    hint = (f"Preserve the existing directory by moving it elsewhere, then rerun: {destination}")
    try:
        metadata = json.loads((destination / "metadata.json").read_text())
        result = json.loads((destination / "report.json").read_text())
        if (metadata.get("settings") != expected or metadata.get("report_sha256") != file_hash(destination / "report.json")
                or not isinstance(result, dict) or not all(k in result for k in required)):
            raise ValueError("metadata or report does not match")
        return result
    except (OSError, ValueError, TypeError) as error:
        raise ValueError(f"Incomplete, legacy, or incompatible evaluation at {destination}. {hint}") from error


## 2. Configuration
Use a new run name for training; existing directories are protected. Device accepts `auto`, `cpu`, `mps`, `cuda`, or a single CUDA index such as `0` or `cuda:0`. Auto prefers CUDA, then MPS supported by the installed Ultralytics/PyTorch combination, then CPU. The selected device is also used for evaluation and visualization.

In [ ]:
PRESET = "smoke"
DATASET_NAME = PRESET
RUN_NAME = PRESET
DEVICE = "auto"

presets = {"smoke": {"epochs": 1}, "development": {"epochs": 100}}
if PRESET not in presets:
    raise ValueError(f"Unknown preset: {PRESET}")
for label, name in (("dataset", DATASET_NAME), ("run", RUN_NAME)):
    if not name or name in {".", ".."} or Path(name).name != name:
        raise ValueError(f"Use a single {label} directory name: {name!r}")

def resolve_device(requested):
    value = str(requested).lower().strip()
    mps_available = bool(TORCH_2_0 and hasattr(torch.backends, "mps") and torch.backends.mps.is_available())
    if value == "auto":
        value = "0" if torch.cuda.is_available() else "mps" if mps_available else "cpu"
    if value == "cpu":
        return "cpu"
    if value in {"mps", "mps:0"}:
        if not mps_available:
            raise RuntimeError("MPS requested but unavailable or unsupported by this Ultralytics/PyTorch version")
        return "mps"
    if value == "cuda":
        value = "0"
    if value.startswith("cuda:"):
        value = value[5:]
    if value.isdigit():
        if not torch.cuda.is_available() or int(value) >= torch.cuda.device_count():
            raise RuntimeError(f"CUDA device {value} requested but unavailable")
        return str(int(value))
    raise ValueError(f"Unsupported device: {requested!r}; choose auto, cpu, mps, or a single CUDA device")

device = resolve_device(DEVICE)
model_name = "yolo26s.pt"
model_path = PROJECT_ROOT / "weights" / model_name
training = dict(imgsz=1280, batch=8, patience=20, workers=4, optimizer="auto",
                seed=42, deterministic=True, pretrained=True,
                hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, translate=0.05, scale=0.1,
                degrees=0.0, shear=0.0, perspective=0.0, flipud=0.0, fliplr=0.0,
                bgr=0.0, mosaic=0.0, mixup=0.0, cutmix=0.0, copy_paste=0.0,
                plots=True, val=True, save=True, **presets[PRESET])
training.update(device=device, amp=device not in {"cpu", "mps"})
get_cfg(overrides=training)
print(f"Preset: {PRESET}; model: {model_name}; epochs: {training['epochs']}; batch: {training['batch']}; image size: {training['imgsz']}")
print(f"Requested device: {DEVICE}; selected device: {device}; requested AMP: {training['amp']}")


The detection policy keeps HSV, small translation and scale changes; geometric flips, rotation and mixing are disabled. Classification-only `auto_augment` and `erasing` do not affect detection. Optional Albumentations introduces extra detection transforms, so training refuses an environment where it is installed. Ultralytics forces workers to zero on CPU and MPS and may disable CUDA AMP after its compatibility check.

## 3. Verify dataset

In [ ]:
def verify_dataset(root):
    root = Path(root).resolve()
    for name in ("config.json", "data.yaml", "manifest.jsonl", "statistics.json"):
        if not (root / name).is_file():
            raise FileNotFoundError(root / name)
    config = json.loads((root / "config.json").read_text())
    for key, expected in {"experiment": "grid-detection", "generator_revision": 7, "status": "complete", "classes": list(YOLO_CLASSES)}.items():
        if config.get(key) != expected:
            raise ValueError(f"{root / 'config.json'}: {key} expected {expected!r}, got {config.get(key)!r}")
    if tuple(YOLO_CLASSES) != ("game_region", "grid", "mine_counter", "reset_button", "timer"):
        raise ValueError("Unexpected production class order")
    data = yaml.safe_load((root / "data.yaml").read_text())
    if not isinstance(data, dict) or data.get("names") != dict(enumerate(YOLO_CLASSES)):
        raise ValueError("Unexpected data.yaml class order")
    path = data.get("path")
    if not isinstance(path, str) or not Path(path).is_absolute() or Path(path).resolve() != root:
        raise ValueError(f"data.yaml must contain the local absolute dataset path {root}; if transferred, update its path manually before continuing")
    splits = ("train", "val", "test")
    stems, seen, identities = {}, {}, {}
    for split in splits:
        if data.get(split) != f"images/{split}":
            raise ValueError(f"Invalid {split} path in data.yaml")
        images = {p.stem for p in (root / "images" / split).glob("*.png")}
        labels = {p.stem for p in (root / "labels" / split).glob("*.txt")}
        count = config.get(split)
        if type(count) is not int or count < 0 or images != labels or len(images) != count:
            raise ValueError(f"Invalid {split} image/label pairing or saved count")
        stems[split], seen[split], identities[split] = images, set(), set()
    records = [json.loads(line) for line in (root / "manifest.jsonl").read_text().splitlines() if line.strip()]
    for record in records:
        split = record.get("split")
        if split not in splits:
            raise ValueError(f"Invalid manifest split: {split!r}")
        image, label = root / record["image"], root / record["label"]
        stem = image.stem
        if (image != root / "images" / split / f"{stem}.png" or label != root / "labels" / split / f"{stem}.txt"
                or stem not in stems[split] or stem in seen[split]):
            raise ValueError(f"Invalid or duplicate manifest entry: {record}")
        seen[split].add(stem)
        identities[split].update(record["board_ids"])
        lines = label.read_text().splitlines()
        if not record["board_ids"] and lines:
            raise ValueError(f"Negative scene has labels: {label}")
        for row in lines:
            fields = row.split()
            if len(fields) != 5 or fields[0] not in {str(i) for i in range(len(YOLO_CLASSES))}:
                raise ValueError(f"Invalid YOLO annotation: {label}: {row}")
            x, y, w, h = map(float, fields[1:])
            if (not all(math.isfinite(v) and 0 <= v <= 1 for v in (x, y, w, h)) or w <= 0 or h <= 0
                    or x-w/2 < -1e-7 or y-h/2 < -1e-7 or x+w/2 > 1+1e-7 or y+h/2 > 1+1e-7):
                raise ValueError(f"Invalid YOLO box: {label}: {row}")
    for split in splits:
        if seen[split] != stems[split]:
            raise ValueError(f"Incomplete manifest coverage: {split}")
    for a, b in combinations(splits, 2):
        if identities[a] & identities[b]:
            raise ValueError(f"Board identities overlap: {a} / {b}")
    print(f"Verified dataset: {root}")
    print("Split counts:", {split: len(stems[split]) for split in splits})
    return config, records

dataset_root = (DATA_DIR / DATASET_NAME).resolve()
dataset_config, records = verify_dataset(dataset_root)
data_yaml = dataset_root / "data.yaml"


## 4. Train model
The directory is reserved before model initialization, and remains reserved after a failure. Native progress and artifacts are retained. To resume separately, load `weights/last.pt`, first verify that `model.ckpt` has a nonnegative epoch and optimizer state and its saved data path matches the original dataset, then use native `model.train(resume=True)`. Completed stripped checkpoints generally cannot resume; this Ultralytics version can otherwise start a new training run instead.

In [ ]:
if importlib.util.find_spec("albumentations") is not None:
    raise RuntimeError("Remove optional Albumentations to preserve the documented augmentation policy")
run_dir = RUNS_DIR / RUN_NAME
run_dir.mkdir(parents=True, exist_ok=False)
options = dict(training, data=str(data_yaml), project=str(RUNS_DIR), name=RUN_NAME, exist_ok=True)
metadata = {
    "dataset_name": DATASET_NAME, "dataset_path": str(dataset_root), "dataset_config": dataset_config,
    "model_initialization": str(model_path), "preset": PRESET, "training_options": options,
    "device_requested": DEVICE, "device_selected": device, "amp_requested": training["amp"],
    "resolved_device": None, "resolved_amp": None, "resolved_workers": None,
    "provenance": run_provenance(PROJECT_ROOT / "src" / "grid_detection", "grid-detection"),
    "versions": {"python": platform.python_version(), "pytorch": str(torch.__version__),
                 "ultralytics": ultralytics.__version__, "cuda_runtime": torch.version.cuda},
}
metadata_path = run_dir / "experiment.json"
def save_metadata():
    metadata_path.write_text(json.dumps(metadata, indent=2) + "\n")
save_metadata()
print(f"Training destination: {run_dir}")
model_path.parent.mkdir(parents=True, exist_ok=True)
model = YOLO(str(model_path))
def capture_resolved(trainer):
    metadata["resolved_device"] = str(trainer.device)
    metadata["resolved_amp"] = bool(trainer.amp)
    metadata["resolved_workers"] = trainer.args.workers
    save_metadata()
model.add_callback("on_train_start", capture_resolved)
model.train(**options)
print(f"Training artifacts: {run_dir}")


## 5. Load saved run
The run's metadata and `args.yaml`, rather than the current preset or dataset selection, determine inspection settings. A relocated dataset needs its saved path updated in `experiment.json` and its `data.yaml` corrected manually. Evaluation requires an intact revision 7 dataset. Native checkpoint and training files remain untouched.

In [ ]:
inspect_dir = RUNS_DIR / RUN_NAME
metadata_path = inspect_dir / "experiment.json"
args_path = inspect_dir / "args.yaml"
for path in (metadata_path, args_path):
    if not path.is_file():
        raise FileNotFoundError(path)
saved = json.loads(metadata_path.read_text())
saved_args = yaml.safe_load(args_path.read_text())
saved_root = Path(saved["dataset_path"]).resolve()
saved_config, saved_records = verify_dataset(saved_root)
if saved_config != saved["dataset_config"]:
    raise ValueError("Current dataset configuration differs from the configuration saved with this run")
saved_yaml = saved_root / "data.yaml"
results_path = inspect_dir / "results.csv"
best_path = inspect_dir / "weights" / "best.pt"
last_path = inspect_dir / "weights" / "last.pt"
with results_path.open(newline="") as f:
    history = [{k.strip(): v for k, v in row.items()} for row in csv.DictReader(f, skipinitialspace=True)]
if not history:
    raise ValueError(f"No training history in {results_path}")
print(f"Run: {inspect_dir}")
print(f"Dataset: {saved_root}")
print(f"Best checkpoint: {best_path} ({'available' if best_path.is_file() else 'missing'})")
print(f"Last checkpoint: {last_path} ({'available' if last_path.is_file() else 'missing'})")
print(f"Recorded epochs: {len(history)}; last epoch: {history[-1]['epoch']}")
print(f"Resolved training: epochs={saved_args.get('epochs')}, batch={saved_args.get('batch')}, image size={saved_args.get('imgsz')}, optimizer={saved_args.get('optimizer')}")
print(f"Device: requested={saved.get('device_requested')}, effective={saved.get('resolved_device')}; AMP: requested={saved.get('amp_requested')}, effective={saved.get('resolved_amp')}; workers={saved.get('resolved_workers')}")


## 6. Analyze training history
The CSV contains validation metrics recorded during training. The exact best epoch cannot reliably be reconstructed from rounded CSV values or stripped checkpoint metadata.

In [ ]:
epochs = [float(row["epoch"]) for row in history]
groups = {
    "Training and validation losses": [k for k in history[0] if k.startswith(("train/", "val/")) and "loss" in k],
    "Precision and recall": [k for k in history[0] if k.startswith(("metrics/precision", "metrics/recall"))],
    "mAP50 and mAP50–95": [k for k in history[0] if k.startswith(("metrics/mAP50(", "metrics/mAP50-95("))],
    "Learning rates": [k for k in history[0] if k.startswith("lr/")],
}
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, (title, keys) in zip(axes.flat, groups.items()):
    for key in keys:
        ax.plot(epochs, [float(row[key]) for row in history], marker="o", markersize=3, label=key)
    ax.set(title=title, xlabel="Epoch")
    ax.xaxis.set_major_locator(MaxNLocator(integer=True, nbins=8))
    if keys:
        ax.legend(fontsize=8)
fig.tight_layout()
plt.show()
if history[-1].get("time"):
    print(f"Reported elapsed training time: {float(history[-1]['time']):.1f} s")
for name in ("confusion_matrix.png", "confusion_matrix_normalized.png", "BoxPR_curve.png", "BoxP_curve.png", "BoxR_curve.png"):
    path = inspect_dir / name
    if path.is_file():
        fig, ax = plt.subplots(figsize=(8, 6))
        ax.imshow(plt.imread(path))
        ax.set_title(name)
        ax.axis("off")
        plt.show()


## 7. Evaluate model
Existing complete, compatible reports are reused. Legacy, incomplete or incompatible directories are never overwritten: move them to a safe location outside `evaluation/val` or `evaluation/test` before evaluating again. Native Ultralytics diagnostics remain in each split directory.


In [ ]:
if not best_path.is_file():
    raise FileNotFoundError(best_path)
evaluation_root = inspect_dir / "evaluation"
reports = {}
for split in ("val", "test"):
    destination = evaluation_root / split
    expected = {"schema": 1, "checkpoint_sha256": file_hash(best_path),
                "dataset_root": str(saved_root), "dataset_config_sha256": file_hash(saved_root / "config.json"),
                "manifest_sha256": file_hash(saved_root / "manifest.jsonl"),
                "data_yaml_sha256": file_hash(saved_yaml),
                "inventory": inventory(list((saved_root / "images" / split).glob("*.png")) +
                                       list((saved_root / "labels" / split).glob("*.txt")), saved_root),
                "split": split, "imgsz": saved_args["imgsz"], "batch": saved_args["batch"],
                "workers": saved_args["workers"], "device": device, "plots": True,
                "ultralytics": ultralytics.__version__}
    if destination.exists():
        report = read_completed(destination, expected, ("aggregate", "per_class", "counts", "timing_ms_per_image"))
        print(f"Loaded evaluation: {destination}")
    else:
        destination.mkdir(parents=True, exist_ok=False)
        print(f"Evaluation destination: {destination}")
        metrics = YOLO(str(best_path)).val(data=str(saved_yaml), split=split, imgsz=saved_args["imgsz"],
                            batch=saved_args["batch"], device=device, workers=saved_args["workers"],
                            project=str(evaluation_root), name=split, exist_ok=True, plots=True)
        box = metrics.box
        classes = {}
        for i, cls in enumerate(box.ap_class_index):
            precision, recall, ap50, ap = box.class_result(i)
            classes[YOLO_CLASSES[int(cls)]] = dict(precision=float(precision), recall=float(recall),
                ap50=float(ap50), ap75=float(box.all_ap[i, 5]), ap50_95=float(ap))
        report = {"aggregate": dict(precision=float(box.mp), recall=float(box.mr), map50=float(box.map50),
                                    map75=float(box.map75), map50_95=float(box.map)),
                  "per_class": classes,
                  "counts": {"images": len(list((saved_root / "images" / split).glob("*.png"))),
                             "instances_per_class": [int(x) for x in metrics.nt_per_class],
                             "images_per_class": [int(x) for x in metrics.nt_per_image]},
                  "timing_ms_per_image": {k: float(v) for k, v in metrics.speed.items()}}
        (destination / "report.json").write_text(json.dumps(report, indent=2, allow_nan=False) + "\n")
        (destination / "metadata.json").write_text(json.dumps({"settings": expected, "report_sha256": file_hash(destination / "report.json")}, indent=2) + "\n")
    reports[split] = report
    a = report["aggregate"]
    print(f"{split}: {report['counts']['images']} images | precision {a['precision']:.2%} | recall {a['recall']:.2%} | mAP50 {a['map50']:.2%} | mAP75 {a['map75']:.2%} | mAP50–95 {a['map50_95']:.2%}")
    for name, values in report["per_class"].items():
        print(f"  {name}: precision {values['precision']:.2%}, recall {values['recall']:.2%}, AP50 {values['ap50']:.2%}, AP75 {values['ap75']:.2%}, AP50–95 {values['ap50_95']:.2%}")
    if not report["per_class"]: print("  Per-class metrics: unavailable")
    print("  Timing (ms/image): " + ", ".join(f"{k} {v:.2f}" for k, v in report["timing_ms_per_image"].items()))


## 8. Visualize predictions
The diagnostic sample uses the validation split, with reproducible random examples, negative scenes, multi-board scenes, smallest labeled grids, and failures found in a bounded random subset. The matching IoU and confidence thresholds below are diagnostic settings, separate from native evaluation metrics. Greedy class-aware one-to-one matching pairs highest-IoU boxes first; it is intended for inspection, not model selection.

In [ ]:
DIAGNOSTIC_SPLIT = "val"
CONFIDENCE = 0.25
MATCH_IOU = 0.5
FAILURE_SCAN_LIMIT = 40
RANDOM_EXAMPLES = 3
if not 0 <= CONFIDENCE <= 1 or not 0 <= MATCH_IOU <= 1 or FAILURE_SCAN_LIMIT < 0:
    raise ValueError("Invalid diagnostic thresholds or scan limit")
if not best_path.is_file():
    raise FileNotFoundError(best_path)
examples = [r for r in saved_records if r["split"] == DIAGNOSTIC_SPLIT]
rng = random.Random(42)
shuffled = rng.sample(examples, len(examples))
scan = shuffled[:FAILURE_SCAN_LIMIT]
print(f"Failure selection scans {len(scan)} of {len(examples)} {DIAGNOSTIC_SPLIT} images (seed 42). Confidence={CONFIDENCE}; matching IoU={MATCH_IOU}.")

def labels_for(record):
    rows = []
    for line in (saved_root / record["label"]).read_text().splitlines():
        cls, x, y, w, h = map(float, line.split())
        rows.append((int(cls), (x-w/2, y-h/2, x+w/2, y+h/2)))
    return rows

def iou(a, b):
    intersection = max(0, min(a[2], b[2])-max(a[0], b[0])) * max(0, min(a[3], b[3])-max(a[1], b[1]))
    union = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - intersection
    return intersection / union if union > 0 else 0.0

def failures(gt, pred):
    pairs = sorted(((iou(gbox, pbox), gi, pi) for gi, (gc, gbox) in enumerate(gt)
                    for pi, (pc, pbox, _) in enumerate(pred) if gc == pc), reverse=True)
    matched_g, matched_p = set(), set()
    for overlap, gi, pi in pairs:
        if overlap >= MATCH_IOU and gi not in matched_g and pi not in matched_p:
            matched_g.add(gi)
            matched_p.add(pi)
    return len(gt)-len(matched_g), len(pred)-len(matched_p)

predictor = YOLO(str(best_path))
prediction_cache = {}
def predictions_for(record):
    key = record["image"]
    if key not in prediction_cache:
        result = predictor.predict(source=str(saved_root / key), imgsz=saved_args["imgsz"], device=device,
                                   conf=CONFIDENCE, verbose=False)[0]
        prediction_cache[key] = [(int(c), tuple(coords), float(score)) for c, coords, score in
                                 zip(result.boxes.cls.cpu().tolist(), result.boxes.xyxyn.cpu().tolist(), result.boxes.conf.cpu().tolist())]
    return prediction_cache[key]

selected = {}
def select(record, reason):
    if record is not None:
        selected.setdefault(record["image"], {"record": record, "reasons": []})["reasons"].append(reason)
for record in shuffled[:RANDOM_EXAMPLES]:
    select(record, "random")
select(next((r for r in shuffled if not r["board_ids"]), None), "negative")
select(next((r for r in shuffled if len(r["board_ids"]) > 1), None), "multi-board")
small = []
for record in examples:
    areas = [(b[2]-b[0])*(b[3]-b[1]) for cls, b in labels_for(record) if YOLO_CLASSES[cls] == "grid"]
    if areas:
        small.append((min(areas), record["image"], record))
if small:
    select(min(small, key=lambda item: (item[0], item[1]))[2], "small grid (relative area)")
missed = false_positive = None
for record in scan:
    miss_count, false_count = failures(labels_for(record), predictions_for(record))
    if miss_count and missed is None:
        missed = record
    if false_count and false_positive is None:
        false_positive = record
select(missed, "missed detection")
select(false_positive, "false positive")
if missed is None: print("No missed detection found in the scanned subset.")
if false_positive is None: print("No false positive found in the scanned subset.")
if not selected: print("No images available for visualization.")

colors = [f"C{i}" for i in range(len(YOLO_CLASSES))]
for item in selected.values():
    record = item["record"]
    with Image.open(saved_root / record["image"]) as image:
        image = image.convert("RGB")
        width, height = image.size
    ground_truth = [(cls, box, None) for cls, box in labels_for(record)]
    predicted = predictions_for(record)
    miss_count, false_count = failures(labels_for(record), predicted)
    print(f"{record['image']}: ground truth={len(ground_truth)}, predictions={len(predicted)}, matched={len(ground_truth)-miss_count}, missed={miss_count}, false positives={false_count}")
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    for ax, boxes, title in zip(axes, (ground_truth, predicted), ("Ground truth", "Predictions")):
        ax.imshow(image)
        label_positions = []
        for cls, (x1, y1, x2, y2), score in sorted(boxes, key=lambda b: b[2] if b[2] is not None else 0, reverse=True):
            color = colors[cls]
            ax.add_patch(Rectangle((x1*width, y1*height), (x2-x1)*width, (y2-y1)*height,
                                   fill=False, edgecolor=color, linewidth=2))
            position = (x1*width, y1*height)
            if score is not None and all(abs(position[0]-x) > width*0.06 or abs(position[1]-y) > height*0.025 for x, y in label_positions):
                ax.text(*position, f"{score:.2f}", color="white", backgroundcolor=color, fontsize=6)
                label_positions.append(position)
        ax.set_title(title)
        ax.axis("off")
    fig.suptitle(f"{record['image']} — {', '.join(item['reasons'])}")
    fig.legend(handles=[Patch(color=colors[i], label=name) for i, name in enumerate(YOLO_CLASSES)],
               loc="lower center", ncol=len(YOLO_CLASSES), fontsize=8)
    fig.tight_layout(rect=(0, 0.05, 1, 0.95))
    plt.show()
